In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


DenseNet121

In [ ]:
# ==========================================
# 1. INSTALL
# ==========================================
!pip install transformers -q

# ==========================================
# 2. IMPORTS
# ==========================================
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torchvision import models, transforms
from transformers import CLIPVisionModel
from tqdm import tqdm

# ==========================================
# 3. PATHS
# ==========================================

VIDEO_ROOT = "/content/drive/MyDrive/Videos_Cropped_2"
WEIGHTS_PATH = "/content/drive/MyDrive/Best_models/best_DenseNet121_v9.pth"
OUTPUT_DIR = "/content/drive/MyDrive/Keyframe_Storage_2/DenseNet121"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 4. CONFIG
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TOP_K = 9
TARGET_FPS = 5

# ==========================================
# 5. MODEL (MATCH TRAINING EXACTLY)
# ==========================================
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.clip = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters():
            p.requires_grad = False

        densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        self.backbone = densenet.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.fc = nn.Linear(1024 + 768, 512)

        self.lstm = nn.LSTM(
            512, 512,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )

        # IMPORTANT: must match training
        self.head = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        b,t,c,h,w = x.shape
        x = x.view(b*t, c, h, w)

        with torch.no_grad():
            cnn = self.pool(self.backbone(x)).flatten(1)
            clip = self.clip(pixel_values=x).pooler_output

        fused = torch.cat([cnn, clip], dim=1)

        x = torch.relu(self.fc(fused))
        x = x.view(b, t, -1)

        x, _ = self.lstm(x)
        return self.head(x).squeeze(-1)

# ==========================================
# 6. LOAD MODEL
# ==========================================
model = Model().to(DEVICE)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

print("Model loaded successfully")

# ==========================================
# 7. PREPROCESS
# ==========================================
normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)

def load_video(path):
    cap = cv2.VideoCapture(path)

    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {path}")
        return None, None, None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if total <= 0 or fps <= 0:
        print(f"[ERROR] Invalid video: {path}")
        cap.release()
        return None, None, None

    interval = max(1, int(fps / TARGET_FPS))
    indices = list(range(0, total, interval))

    frames_resized = []
    frames_original = []

    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()

        if not ret:
            continue

        frames_original.append(frame.copy())

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, (224,224))
        frames_resized.append(frame_resized)

    cap.release()

    if len(frames_resized) == 0:
        print(f"[WARNING] No frames extracted: {path}")
        return None, None, None

    vid = torch.from_numpy(np.array(frames_resized)).permute(0,3,1,2).float()/255.0
    vid = normalize(vid)

    return vid.unsqueeze(0).to(DEVICE), frames_original, indices

# ==========================================
# 8. KEYFRAME EXTRACTION
# ==========================================
def extract_keyframes(video_path):

    vid_tensor, raw_frames, indices = load_video(video_path)

    if vid_tensor is None:
        return

    if len(raw_frames) == 0 or len(indices) == 0:
        print(f"[WARNING] Empty frames: {video_path}")
        return

    with torch.no_grad():
        out = model(vid_tensor)
        probs = torch.sigmoid(out).cpu().numpy()[0]

    if len(probs) == 0:
        print(f"[WARNING] Empty predictions: {video_path}")
        return

    k = min(TOP_K, len(probs))
    top_idx = np.argsort(probs)[-k:]

    # Build structured output path
    word = os.path.basename(os.path.dirname(video_path))
    video_name = os.path.splitext(os.path.basename(video_path))[0]

    save_dir = os.path.join(OUTPUT_DIR, word, video_name)
    os.makedirs(save_dir, exist_ok=True)

    saved_count = 0

    for rank, idx in enumerate(top_idx):

        if idx >= len(raw_frames) or idx >= len(indices):
            continue

        frame = raw_frames[idx]
        frame_id = int(indices[idx])

        save_path = os.path.join(save_dir, f"kf_{rank}_frame_{frame_id}.jpg")

        success = cv2.imwrite(save_path, frame)

        if success:
            saved_count += 1
        else:
            print(f"[ERROR] Failed to save: {save_path}")

    if saved_count == 0:
        print(f"[WARNING] No frames saved for: {video_name}")
    else:
        print(f"[OK] {saved_count} frames saved for {video_name}")

# ==========================================
# 9. COLLECT VIDEOS
# ==========================================
video_list = []

for word in os.listdir(VIDEO_ROOT):
    word_path = os.path.join(VIDEO_ROOT, word)

    if not os.path.isdir(word_path):
        continue

    for v in os.listdir(word_path):
        if v.endswith(".mp4"):
            video_list.append(os.path.join(word_path, v))

print(f"Total videos found: {len(video_list)}")

# ==========================================
# 10. RUN
# ==========================================
for vid in tqdm(video_list):
    extract_keyframes(vid)

print("Keyframe extraction completed")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[OK] 9 frames saved for Anugraham_7


100%|█████████▉| 219/220 [1:52:50<00:16, 16.07s/it]

[OK] 9 frames saved for Anugraham_8


100%|██████████| 220/220 [1:53:07<00:00, 30.85s/it]

[OK] 9 frames saved for Anugraham_9
Keyframe extraction completed


ResNet50

In [ ]:
# ==========================================
# 1. INSTALL
# ==========================================
!pip install transformers -q

# ==========================================
# 2. IMPORTS
# ==========================================
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torchvision import models, transforms
from transformers import CLIPVisionModel
from tqdm import tqdm

# ==========================================
# 3. PATHS
# ==========================================
#VIDEO_ROOT = "/content/drive/MyDrive/Kathakali_Videos_23rd_Jan_2026/Videos_Cropped"
VIDEO_ROOT = "/content/drive/MyDrive/Videos_Cropped_2"
WEIGHTS_PATH = "/content/drive/MyDrive/Best_models/best_ResNet50_v9.pth"
OUTPUT_DIR = "/content/drive/MyDrive/Keyframe_Storage_2/ResNet50"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 4. CONFIG
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TOP_K = 9
TARGET_FPS = 5

# ==========================================
# 5. MODEL (ResNet50)
# ==========================================
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        # CLIP
        self.clip = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters():
            p.requires_grad = False

        # ResNet50 backbone (2048-dim)
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # remove FC

        # Fusion
        self.fc = nn.Linear(2048 + 768, 512)

        # Temporal modeling
        self.lstm = nn.LSTM(
            512, 512,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )

        # Head (must match training)
        self.head = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)

        with torch.no_grad():
            cnn = self.backbone(x).view(x.size(0), -1)   # (B*T, 2048)
            clip = self.clip(pixel_values=x).pooler_output  # (B*T, 768)

        fused = torch.cat([cnn, clip], dim=1)

        x = torch.relu(self.fc(fused))
        x = x.view(b, t, -1)

        x, _ = self.lstm(x)
        return self.head(x).squeeze(-1)

# ==========================================
# 6. LOAD MODEL
# ==========================================
model = Model().to(DEVICE)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

print("ResNet50 model loaded successfully")

# ==========================================
# 7. PREPROCESS
# ==========================================
normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)

def load_video(path):
    cap = cv2.VideoCapture(path)

    if not cap.isOpened():
        print(f"[ERROR] Cannot open: {path}")
        return None, None, None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if total <= 0 or fps <= 0:
        cap.release()
        return None, None, None

    interval = max(1, int(fps / TARGET_FPS))
    indices = list(range(0, total, interval))

    frames_resized = []
    frames_original = []

    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()

        if not ret:
            continue

        frames_original.append(frame.copy())

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, (224,224))
        frames_resized.append(frame_resized)

    cap.release()

    if len(frames_resized) == 0:
        return None, None, None

    vid = torch.from_numpy(np.array(frames_resized)).permute(0,3,1,2).float()/255.0
    vid = normalize(vid)

    return vid.unsqueeze(0).to(DEVICE), frames_original, indices

# ==========================================
# 8. KEYFRAME EXTRACTION
# ==========================================
def extract_keyframes(video_path):

    vid_tensor, raw_frames, indices = load_video(video_path)

    if vid_tensor is None:
        return

    with torch.no_grad():
        out = model(vid_tensor)
        probs = torch.sigmoid(out).cpu().numpy()[0]

    k = min(TOP_K, len(probs))
    top_idx = np.argsort(probs)[-k:]

    word = os.path.basename(os.path.dirname(video_path))
    video_name = os.path.splitext(os.path.basename(video_path))[0]

    save_dir = os.path.join(OUTPUT_DIR, word, video_name)
    os.makedirs(save_dir, exist_ok=True)

    for rank, idx in enumerate(top_idx):
        if idx >= len(raw_frames):
            continue

        frame = raw_frames[idx]
        frame_id = int(indices[idx])

        save_path = os.path.join(save_dir, f"kf_{rank}_frame_{frame_id}.jpg")
        cv2.imwrite(save_path, frame)

# ==========================================
# 9. COLLECT VIDEOS
# ==========================================
video_list = []

for word in os.listdir(VIDEO_ROOT):
    word_path = os.path.join(VIDEO_ROOT, word)

    if not os.path.isdir(word_path):
        continue

    for v in os.listdir(word_path):
        if v.endswith(".mp4"):
            video_list.append(os.path.join(word_path, v))

print(f"Total videos: {len(video_list)}")

# ==========================================
# 10. RUN
# ==========================================
for vid in tqdm(video_list):
    extract_keyframes(vid)

print("Keyframe extraction completed")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.embeddings.token_embedding.weight                 | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_projection.weight                                       | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight    

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 52.9MB/s]


ResNet50 model loaded successfully
Total videos: 220


100%|██████████| 220/220 [1:49:59<00:00, 30.00s/it]

Keyframe extraction completed


EfficientNetB0

In [ ]:
# ==========================================
# 1. INSTALL
# ==========================================
!pip install transformers -q

# ==========================================
# 2. IMPORTS
# ==========================================
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torchvision import models, transforms
from transformers import CLIPVisionModel
from tqdm import tqdm

# ==========================================
# 3. PATHS (UPDATE IF NEEDED)
# ==========================================
#VIDEO_ROOT = "/content/drive/MyDrive/Kathakali_Videos_23rd_Jan_2026/Videos_Cropped"
VIDEO_ROOT = "/content/drive/MyDrive/Videos_Cropped_2"
WEIGHTS_PATH = "/content/drive/MyDrive/Best_models/best_EfficientNet_v9.pth"
OUTPUT_DIR = "/content/drive/MyDrive/Keyframe_Storage_2/EfficientNet"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 4. CONFIG
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TOP_K = 9
TARGET_FPS = 5

# ==========================================
# 5. MODEL (EfficientNet-B0)
# ==========================================
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        # CLIP
        self.clip = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters():
            p.requires_grad = False

        # EfficientNet-B0 backbone (1280)
        efficientnet = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )
        self.backbone = efficientnet.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        # Fusion
        self.fc = nn.Linear(1280 + 768, 512)

        # Temporal modeling
        self.lstm = nn.LSTM(
            512, 512,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )

        # Head (must match training)
        self.head = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)

        with torch.no_grad():
            cnn = self.pool(self.backbone(x)).view(x.size(0), -1)
            clip = self.clip(pixel_values=x).pooler_output

        fused = torch.cat([cnn, clip], dim=1)

        x = torch.relu(self.fc(fused))
        x = x.view(b, t, -1)

        x, _ = self.lstm(x)
        return self.head(x).squeeze(-1)

# ==========================================
# 6. LOAD MODEL
# ==========================================
model = Model().to(DEVICE)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

print("EfficientNet-B0 model loaded")

# ==========================================
# 7. PREPROCESS
# ==========================================
normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)

def load_video(path):
    cap = cv2.VideoCapture(path)

    if not cap.isOpened():
        print(f"[ERROR] Cannot open: {path}")
        return None, None, None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if total <= 0 or fps <= 0:
        cap.release()
        return None, None, None

    interval = max(1, int(fps / TARGET_FPS))
    indices = list(range(0, total, interval))

    frames_resized = []
    frames_original = []

    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()

        if not ret:
            continue

        frames_original.append(frame.copy())

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, (224,224))
        frames_resized.append(frame_resized)

    cap.release()

    if len(frames_resized) == 0:
        return None, None, None

    vid = torch.from_numpy(np.array(frames_resized)).permute(0,3,1,2).float()/255.0
    vid = normalize(vid)

    return vid.unsqueeze(0).to(DEVICE), frames_original, indices

# ==========================================
# 8. KEYFRAME EXTRACTION
# ==========================================
def extract_keyframes(video_path):

    vid_tensor, raw_frames, indices = load_video(video_path)

    if vid_tensor is None:
        return

    with torch.no_grad():
        out = model(vid_tensor)
        probs = torch.sigmoid(out).cpu().numpy()[0]

    k = min(TOP_K, len(probs))
    top_idx = np.argsort(probs)[-k:]

    word = os.path.basename(os.path.dirname(video_path))
    video_name = os.path.splitext(os.path.basename(video_path))[0]

    save_dir = os.path.join(OUTPUT_DIR, word, video_name)
    os.makedirs(save_dir, exist_ok=True)

    for rank, idx in enumerate(top_idx):
        if idx >= len(raw_frames):
            continue

        frame = raw_frames[idx]
        frame_id = int(indices[idx])

        save_path = os.path.join(save_dir, f"kf_{rank}_frame_{frame_id}.jpg")
        cv2.imwrite(save_path, frame)

# ==========================================
# 9. COLLECT VIDEOS
# ==========================================
video_list = []

for word in os.listdir(VIDEO_ROOT):
    word_path = os.path.join(VIDEO_ROOT, word)

    if not os.path.isdir(word_path):
        continue

    for v in os.listdir(word_path):
        if v.endswith(".mp4"):
            video_list.append(os.path.join(word_path, v))

print(f"Total videos: {len(video_list)}")

# ==========================================
# 10. RUN
# ==========================================
for vid in tqdm(video_list):
    extract_keyframes(vid)

print("Keyframe extraction completed")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.embeddings.token_embedding.weight                 | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_projection.weight                                       | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight    

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 46.4MB/s]


EfficientNet-B0 model loaded
Total videos: 220


100%|██████████| 220/220 [1:24:11<00:00, 22.96s/it]

Keyframe extraction completed


MobileNetv2

In [ ]:
# ==========================================
# 1. INSTALL
# ==========================================
!pip install transformers -q

# ==========================================
# 2. IMPORTS
# ==========================================
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torchvision import models, transforms
from transformers import CLIPVisionModel
from tqdm import tqdm

# ==========================================
# 3. PATHS (UPDATE IF NEEDED)
# ==========================================
#VIDEO_ROOT = "/content/drive/MyDrive/Kathakali_Videos_23rd_Jan_2026/Videos_Cropped"
VIDEO_ROOT = "/content/drive/MyDrive/Videos_Cropped_2"
WEIGHTS_PATH = "/content/drive/MyDrive/Best_models/best_MobileNetV2_v9.pth"
OUTPUT_DIR = "/content/drive/MyDrive/Keyframe_Storage_2/MobileNetV2"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 4. CONFIG
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TOP_K = 9
TARGET_FPS = 5

# ==========================================
# 5. MODEL (MobileNetV2)
# ==========================================
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        # CLIP
        self.clip = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters():
            p.requires_grad = False

        # MobileNetV2 backbone (1280-dim)
        mobilenet = models.mobilenet_v2(
            weights=models.MobileNet_V2_Weights.DEFAULT
        )
        self.backbone = mobilenet.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        # Fusion
        self.fc = nn.Linear(1280 + 768, 512)

        # Temporal modeling
        self.lstm = nn.LSTM(
            512, 512,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )

        # Head (must match training)
        self.head = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)

        with torch.no_grad():
            cnn = self.pool(self.backbone(x)).view(x.size(0), -1)   # (B*T, 1280)
            clip = self.clip(pixel_values=x).pooler_output          # (B*T, 768)

        fused = torch.cat([cnn, clip], dim=1)

        x = torch.relu(self.fc(fused))
        x = x.view(b, t, -1)

        x, _ = self.lstm(x)
        return self.head(x).squeeze(-1)

# ==========================================
# 6. LOAD MODEL
# ==========================================
model = Model().to(DEVICE)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

print("MobileNetV2 model loaded")

# ==========================================
# 7. PREPROCESS
# ==========================================
normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)

def load_video(path):
    cap = cv2.VideoCapture(path)

    if not cap.isOpened():
        print(f"[ERROR] Cannot open: {path}")
        return None, None, None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if total <= 0 or fps <= 0:
        cap.release()
        return None, None, None

    interval = max(1, int(fps / TARGET_FPS))
    indices = list(range(0, total, interval))

    frames_resized = []
    frames_original = []

    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()

        if not ret:
            continue

        frames_original.append(frame.copy())

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, (224,224))
        frames_resized.append(frame_resized)

    cap.release()

    if len(frames_resized) == 0:
        return None, None, None

    vid = torch.from_numpy(np.array(frames_resized)).permute(0,3,1,2).float()/255.0
    vid = normalize(vid)

    return vid.unsqueeze(0).to(DEVICE), frames_original, indices

# ==========================================
# 8. KEYFRAME EXTRACTION
# ==========================================
def extract_keyframes(video_path):

    vid_tensor, raw_frames, indices = load_video(video_path)

    if vid_tensor is None:
        return

    with torch.no_grad():
        out = model(vid_tensor)
        probs = torch.sigmoid(out).cpu().numpy()[0]

    k = min(TOP_K, len(probs))
    top_idx = np.argsort(probs)[-k:]

    word = os.path.basename(os.path.dirname(video_path))
    video_name = os.path.splitext(os.path.basename(video_path))[0]

    save_dir = os.path.join(OUTPUT_DIR, word, video_name)
    os.makedirs(save_dir, exist_ok=True)

    for rank, idx in enumerate(top_idx):
        if idx >= len(raw_frames):
            continue

        frame = raw_frames[idx]
        frame_id = int(indices[idx])

        save_path = os.path.join(save_dir, f"kf_{rank}_frame_{frame_id}.jpg")
        cv2.imwrite(save_path, frame)

# ==========================================
# 9. COLLECT VIDEOS
# ==========================================
video_list = []

for word in os.listdir(VIDEO_ROOT):
    word_path = os.path.join(VIDEO_ROOT, word)

    if not os.path.isdir(word_path):
        continue

    for v in os.listdir(word_path):
        if v.endswith(".mp4"):
            video_list.append(os.path.join(word_path, v))

print(f"Total videos: {len(video_list)}")

# ==========================================
# 10. RUN
# ==========================================
for vid in tqdm(video_list):
    extract_keyframes(vid)

print("Keyframe extraction completed")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.embeddings.token_embedding.weight                 | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_projection.weight                                       | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight    

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 32.8MB/s]


MobileNetV2 model loaded
Total videos: 220


100%|██████████| 220/220 [1:21:13<00:00, 22.15s/it]

Keyframe extraction completed
